In [2]:
### PFDF Documentation : <https://ghsc.code-pages.usgs.gov/lhp/pfdf/api/data/noaa/atlas14.html> ###
!pip install pfdf -i https://code.usgs.gov/api/v4/groups/859/-/packages/pypi/simple


Looking in indexes: https://code.usgs.gov/api/v4/groups/859/-/packages/pypi/simple


In [3]:
from google.colab import drive
drive.mount('/content/drive')

!git config --global user.email "ezuetell@andrew.cmu.edu"
!git config --global user.name "emzu"

try:
    !git clone "https://github.com/emzu/futureIDF"
except:
    print("Already cloned")

%cd /content/futureIDF
!git pull

# Load Packages
!pip install -r requirements.txt

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
fatal: destination path 'futureIDF' already exists and is not an empty directory.
/content/futureIDF
Already up to date.


In [4]:
import os
import geopandas as gpd
import pandas as pd
import pfdf
from pfdf.data.noaa import atlas14
import time

In [7]:
!git pull

# Import modules
import sys
import importlib
# Ensure workspace root is on sys.path so the local `modules` package can be imported
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import glob
import os
import warnings
warnings.filterwarnings('ignore')

from modules import config, data_io, timeseries, plotting, process_rp, geospatial
importlib.reload(timeseries)
importlib.reload(data_io)
importlib.reload(process_rp)
importlib.reload(geospatial)

remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 9 (delta 0), reused 9 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (9/9), 9.78 MiB | 6.54 MiB/s, done.
From https://github.com/emzu/futureIDF
   113e3b7..f59568f  main       -> origin/main
Updating 113e3b7..f59568f
Fast-forward
 data/Boundaries/MARISA_domain.cpg |   1 +
 data/Boundaries/MARISA_domain.dbf | Bin 0 -> 42280 bytes
 data/Boundaries/MARISA_domain.prj |   1 +
 data/Boundaries/MARISA_domain.shp | Bin 0 -> 14124060 bytes
 data/Boundaries/MARISA_domain.shx | Bin 0 -> 2196 bytes
 5 files changed, 2 insertions(+)
 create mode 100644 data/Boundaries/MARISA_domain.cpg
 create mode 100644 data/Boundaries/MARISA_domain.dbf
 create mode 100644 data/Boundaries/MARISA_domain.prj
 create mode 100644 data/Boundaries/MARISA_domain.shp
 create mode 100644 data/Boundaries/MARISA_domain.shx


<module 'modules.geospatial' from '/content/futureIDF/modules/geospatial.py'>

In [8]:
counties_gdf = data_io.load_counties()
# Create lists to store results
results = []

In [33]:
# Create output directory
out_dir = 'content/atlas14_data'
os.makedirs(out_dir, exist_ok=True)

# Store 24-hr data for all counties
county_24hr_data = pd.DataFrame()

# Download for each county
for idx, county in counties_gdf[:1].iterrows():
    try:
        centroid = county.geometry.centroid
        lon, lat = centroid.x, centroid.y

        county_name = county.get('NAME')
        state_name = county.get('STATE')

        csv_path = atlas14.download(
            lat=lat,
            lon=lon,
            series='pds',
            statistic='all',
            data='intensity',
            units='english',
            parent='content/atlas14_data',
            name=f"{county_name}_{state_name}_{idx}.csv",
            overwrite=False
        )

        # Read the CSV
        df = pd.read_csv(csv_path, header = 11)
        df.set_index('by duration for ARI (years):', inplace=True)

        data_24hr = df.loc['24-hr:']
        # Add county info
        data_24hr = data_24hr.copy()*24
        data_24hr['county_name'] = county_name
        data_24hr['county_fips'] = county.get('FIPS', None)
        data_24hr['lon'] = lon
        data_24hr['lat'] = lat
        data_24hr['idx_ref'] = idx
        data_24hr.index = pd.Index(['mean', 'upper', 'lower'], name='CI')

        county_24hr_data = pd.concat([county_24hr_data, data_24hr], axis=1)

        print(f"Downloaded {county_name}: extracted {len(data_24hr)} 24-hr records")
        time.sleep(1)

    except Exception as e:
        print(f"Error for county {county.get('NAME', idx)}: {e}")
        continue

# Combine all 24-hr data
all_24hr = county_24hr_data.set_index('county_name')
print(f"\nTotal 24-hr records: {len(all_24hr)}")

Downloaded Beaver: extracted 3 24-hr records

Total 24-hr records: 3


In [34]:
county_24hr_data

,1,2,5,10,25,50,100,200,500,1000,county_name,county_fips,lon,lat
CI,,,,,,,,,,,,,,
mean,1.968,2.352,2.880,3.288,3.912,4.392,4.920,5.448,6.216,6.816,Beaver,None,-80.349295,40.682262
upper,2.112,2.520,3.096,3.552,4.176,4.704,5.232,5.808,6.600,7.224,Beaver,None,-80.349295,40.682262
lower,1.848,2.184,2.688,3.072,3.624,4.056,4.512,4.992,5.664,6.168,Beaver,None,-80.349295,40.682262


In [ ]:
all_24hr.reset_index(inplace=True)
all_24hr.rename(columns={'index': 'county_name'}, inplace=True)

In [ ]:
all_24hr

,county_name,1,2,5,10,25,50,100,200,500,1000,county_fips,lon,lat
0,Beaver,1.968,2.352,2.88,3.288,3.912,4.392,4.92,5.448,6.216,6.816,None,-80.349295,40.682262
1,Salem,2.496,3.024,3.84,4.512,5.472,6.288,7.176,8.136,9.504,10.656,None,-80.055475,37.286403
2,Montgomery,2.28,2.784,3.528,4.152,5.04,5.784,6.6,7.464,8.736,9.768,None,-80.387215,37.174291
3,Poquoson,2.904,3.552,4.584,5.496,6.84,7.992,9.288,10.752,12.936,14.832,None,-76.2705,37.150278
4,James City,2.928,3.552,4.584,5.472,6.792,7.92,9.168,10.584,12.672,14.448,None,-76.773867,37.313338
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
257,Ontario,1.92,2.28,2.856,3.336,3.984,4.464,5.016,5.664,6.72,7.608,None,-77.481723,42.588807
258,Madison,2.04,2.424,3.072,3.6,4.32,4.848,5.424,6.096,7.056,7.872,None,-75.57269,42.812959
259,Onondaga,2.04,2.424,3.072,3.6,4.32,4.872,5.424,6.072,7.008,7.752,None,-76.025977,42.822561
260,Oneida,2.112,2.496,3.12,3.624,4.32,4.848,5.4,6.048,6.96,7.728,None,-75.309267,42.901726


In [ ]:
all_24hr.to_parquet('atlas14_24hr_data.parquet', index=False)